In [0]:
# MAGIC %md
# MAGIC # Bronze Layer: Automated Batch Ingestion Engine
# MAGIC 
# MAGIC - **Sources**: `abfss://landing@sadataproject01.dfs.core.windows.net/raw_feeds/`
# MAGIC - **Target**: Managed Delta Tables in `bronze` schema
# MAGIC - **Metadata Audit Columns**:
# MAGIC    - `_ingested_at`: UTC timestamp of ingestion
# MAGIC    - `_source_file`: File path extracted via `_metadata.file_path`

# COMMAND ----------

from pyspark.sql import functions as F
import re

# Ensure bronze schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Define landing base path
landing_base = "abfss://landing@sadataproject01.dfs.core.windows.net/raw_feeds/"

# Folders to scan
source_folders = [
    "PRJ_ECom_Dataset_1/PRJ_ECom_Dataset_1/",
    "PRJ_ECom_Dataset_1/PRJ_ECom_Dataset_2/",
    "PRJ_ECom_Dataset_2/"
]

In [0]:


# COMMAND ----------

def clean_table_name(filename):
    """Derives a clean Delta table name from raw CSV file naming conventions."""
    name = filename.replace(".csv", "").lower()
    # Strip common project prefixes
    name = re.sub(r"^prj_ecom_", "", name)
    # Strip dataset slice suffixes like _data_1, _data_2, _1, _2
    name = re.sub(r"(_data)?_\d+$", "", name)
    return name



In [0]:
# COMMAND ----------

# Gather all CSV files across source directories
csv_sources = {}

for folder in source_folders:
    folder_path = f"{landing_base}{folder}"
    try:
        files = dbutils.fs.ls(folder_path)
        for f in files:
            if f.name.endswith(".csv") and not f.name.startswith("_"):
                table_name = clean_table_name(f.name)
                if table_name not in csv_sources:
                    csv_sources[table_name] = []
                csv_sources[table_name].append(f.path)
    except Exception as e:
        print(f"Skipping inaccessible path {folder_path}: {e}")

print(f"Discovered {len(csv_sources)} distinct entity tables to ingest:")
for t, paths in sorted(csv_sources.items()):
    print(f" - bronze.{t} ({len(paths)} file slices)")



In [0]:
# COMMAND ----------

# Batch Ingestion Loop
success_count = 0
failed_tables = []

for table_name, file_paths in sorted(csv_sources.items()):
    print(f"Ingesting bronze.{table_name}...")
    try:
        # Read with schema inference and auto-populated metadata
        df_raw = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_paths)
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
        )
        
        # Save as managed Delta table in Unity Catalog
        (
            df_raw.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"bronze.{table_name}")
        )
        
        row_cnt = spark.table(f"bronze.{table_name}").count()
        print(f"  --> Saved bronze.{table_name} | Rows: {row_cnt}")
        success_count += 1
    except Exception as e:
        print(f"  [ERROR] Failed to ingest {table_name}: {e}")
        failed_tables.append((table_name, str(e)))

print(f"\n--- Batch Ingestion Complete: {success_count} tables loaded, {len(failed_tables)} failed ---")

In [0]:
# Verify queries run instantly against your Bronze layer
display(spark.sql("SELECT count(*) AS user_count FROM bronze.user"))
display(spark.sql("SELECT count(*) AS addr_count FROM bronze.user_addr_scd2"))